In [2]:
import pandas as pd 


In [3]:
df = pd.read_csv("Dirty_Housing_Regression.csv")

In [4]:
df.head()

,city,year,month,sales,volume,median_sale_price,listings,inventory,date,property_inspection_note,external_market_rating,temporary_file_id
0,Garland,2003,1,151.0,16880476.0,108700.0,1110.0,4.9,2003.000000,NaN,NaN,H-01532
1,Collin County,2010,4,1254.0,290057387.0,193400.0,5041.0,5.1,2010.250000,noted,NaN,H-01921
2,Paris,2006,4,36.0,3365000.0,86000.0,NaN,NaN,2006.250000,noted,NaN,H-01674
3,Bryan-College Station,2014,10,218.0,46484824.0,176100.0,1031.0,NaN,2014.750000,NaN,NaN,H-02725
4,McAllen,2001,2,107.0,11070000.0,72800.0,NaN,NaN,2001.083333,NaN,NaN,H-00653


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3035 entries, 0 to 3034
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   city                      3035 non-null   str    
 1   year                      3035 non-null   int64  
 2   month                     3035 non-null   int64  
 3   sales                     2825 non-null   str    
 4   volume                    2853 non-null   str    
 5   median_sale_price         3035 non-null   float64
 6   listings                  2458 non-null   float64
 7   inventory                 2408 non-null   float64
 8   date                      3035 non-null   float64
 9   property_inspection_note  124 non-null    str    
 10  external_market_rating    196 non-null    float64
 11  temporary_file_id         3035 non-null   str    
dtypes: float64(5), int64(2), str(5)
memory usage: 378.4 KB


In [6]:
df.sales =  pd.to_numeric(df.sales, errors='coerce')
df.volume = pd.to_numeric(df.volume,errors='coerce')

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3035 entries, 0 to 3034
Data columns (total 12 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   city                      3035 non-null   str    
 1   year                      3035 non-null   int64  
 2   month                     3035 non-null   int64  
 3   sales                     2801 non-null   float64
 4   volume                    2834 non-null   float64
 5   median_sale_price         3035 non-null   float64
 6   listings                  2458 non-null   float64
 7   inventory                 2408 non-null   float64
 8   date                      3035 non-null   float64
 9   property_inspection_note  124 non-null    str    
 10  external_market_rating    196 non-null    float64
 11  temporary_file_id         3035 non-null   str    
dtypes: float64(7), int64(2), str(3)
memory usage: 336.4 KB


In [8]:
df = df.dropna(subset=['sales'])

In [9]:
features = [
    'city', 'year', 'month',  'volume', 'median_sale_price',
       'listings', 'inventory'
]
X = df[features]
y = df.sales



In [10]:
numerical_features = [
     'year', 'month',  'volume', 'median_sale_price',
       'listings', 'inventory'

]

categorical_features = [
    'city'
]



In [11]:
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

numerical_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
]
)
categorical_pipeline =  Pipeline([
    ('imputer',SimpleImputer(strategy='most_frequent')),
        ('encoder',OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num',numerical_pipeline,numerical_features),
    ('cat',categorical_pipeline,categorical_features)
])

models = {
    'Linear Regression' : (
        LinearRegression(),{
            'model__fit_intercept' :[True,False]
        }
    ),
    'Ridge Regression':(
        Ridge(),
        {
            'model__alpha':[0.01,0.1,1,10,100]
        }
    ),
    'Random Forest':(
        RandomForestRegressor(random_state=42),{
             'model__n_estimators': [100, 200],
            'model__max_depth': [None, 5, 10],
            'model__min_samples_split': [2, 5]
        }
    ),
   'Decision Tree':(
       DecisionTreeRegressor(random_state=42),
       {
           'model__max_depth': [None, 5, 10],
            'model__min_samples_split': [2, 5],
            'model__min_samples_leaf': [1, 2]
       }
   )
}


from sklearn.pipeline import Pipeline

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
cv = KFold(n_splits=5, shuffle=True, random_state=42)

results = []
best_models = {}

for name, ( regressor , param_grid) in models.items():
    Pipe = Pipeline([
        ('preprocessing',preprocessor),
        ('model',regressor)
    ])

    grid = GridSearchCV(
        Pipe,
        param_grid ,
        cv = cv ,
        scoring='r2' ,
        n_jobs=-1

    )

    grid.fit(X_train, y_train)

    best_models[name] = grid.best_estimator_


    results.append({
        'Model': name,
        'Best CV R2': grid.best_score_,
        'Best Parameters': grid.best_params_
    })


results_df = pd.DataFrame(results)
print(results_df)

   
best_name = results_df.loc[
    results_df['Best CV R2'].idxmax(), 'Model'
]

best_model = best_models[best_name]

print("Best Model:", best_name)

# 10. Final test evaluation (only once)
y_pred = best_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2 Score:", r2_score(y_test, y_pred))


               Model  Best CV R2  \
0  Linear Regression    0.956057   
1   Ridge Regression    0.956130   
2      Random Forest    0.967978   
3      Decision Tree    0.951376   

                                     Best Parameters  
0                     {'model__fit_intercept': True}  
1                              {'model__alpha': 0.1}  
2  {'model__max_depth': None, 'model__min_samples...  
3  {'model__max_depth': 10, 'model__min_samples_l...  
Best Model: Random Forest
MAE: 38.100142602495545
MSE: 14019.673136898395
RMSE: 118.40470065372571
R2 Score: 0.9884210975666361
